# 02 — Preparação das variáveis

Primeiro são calculados, por ativo, retornos defasados, volatilidade realizada,
Garman–Klass, GARCH(1,1), assimetria, curtose e spread de Corwin–Schultz,
conforme a disponibilidade dos dados. O GARCH tem parâmetros reestimados
periodicamente e variância atualizada a cada dia com os retornos já observados.
O VIX e a volatilidade implícita do Ibovespa completam o conjunto de entradas.
As variáveis são salvas em `data/processed/features.parquet`, antes da seleção.

Depois são feitos os diagnósticos de estacionariedade, correlação e VIF.
ADF e KPSS são aplicados separadamente a cada ativo; correlação e VIF usam
o painel empilhado. Esses resultados descrevem a amostra completa.
A seleção usada na previsão é refeita no treino de cada janela walk-forward.

O painel segue os pregões da B3. Quando a B3 abre e um mercado estrangeiro
permanece fechado, a ausência segue a regra de preenchimento causal de até
três pregões. As lacunas restantes e as ausências iniciais necessárias ao
cálculo das variáveis são preservadas.

Soja e café usam os índices ajustados por rolagem do pysystemtrade. Os preços
são convertidos de dólares para reais com o câmbio do mesmo dia, antes do
cálculo das variáveis. Seus retornos e volatilidades, portanto, incorporam
a variação cambial.

## Séries de futuros

`scripts/build_public_futures.py` reconstrói soja e café a partir da base pública
do pysystemtrade, fixada por commit e conferida por SHA256. Na rolagem, o ajuste
usa a razão entre os preços dos dois contratos na mesma data. A soja segue o
contrato de novembro, com rolagem em setembro; o café segue os vencimentos de
março, maio, julho, setembro e dezembro.

As séries contínuas do Yahoo e os genéricos `S 1 COMB` e `KC1 COMB` do Bloomberg
mantêm descontinuidades de rolagem. Na soja, por exemplo, a queda próxima de 23%
em 15/09/2008 fica perto de 2% após o ajuste. Movimentos do café associados à
seca de 2014 e às geadas de julho de 2021 permanecem na série.

A base contém apenas fechamento. Garman–Klass, Corwin–Schultz e volume são
excluídos para os futuros. O histórico do café termina em 28/03/2024 e limita
o fim da amostra comum.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.features import align_and_fill_ohlc, build_asset_features, convert_ohlc_to_brl, get_b3_trading_days, equal_weighted_portfolio_return
from src.data.feature_selection import correlation_matrix, stationarity_report, vif_filter
from src.data.collectors import load_public_futures_close

config = yaml.safe_load((PROJECT_ROOT / "config.yaml").read_text(encoding="utf-8"))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## Alinhamento das séries e tratamento de ausências

O painel começa na primeira data comum aos sete ativos e termina no fim do
histórico do café. Os indicadores do CEPEA não entram nessa composição.

In [2]:
equities_fx = pd.read_parquet(RAW_DIR / "equities_fx.parquet")
implied_vol = pd.read_parquet(RAW_DIR / "implied_vol.parquet")

PUBLIC_FUTURES_FILES = {
    "ZS=F": RAW_DIR / "futures_public" / "SOYBEAN_daily.parquet",
    "KC=F": RAW_DIR / "futures_public" / "COFFEE_daily.parquet",
}

equity_fx_tickers = [a["ticker"] for a in config["assets"] if a["category"] in ("equity", "fx")]
futures_tickers = [a["ticker"] for a in config["assets"] if a["category"] == "commodity"]

core_ohlc = {t: equities_fx[t] for t in equity_fx_tickers}
core_ohlc.update({t: load_public_futures_close(PUBLIC_FUTURES_FILES[t]) for t in futures_tickers})

# A carteira exige os sete ativos todo dia: o painel vai do início do ativo mais
# tardio até o fim do ativo que acaba primeiro (café, 28/03/2024).
common_start = max(df["Close"].dropna().index.min() for df in core_ohlc.values())
end = min(min(df["Close"].dropna().index.max() for df in core_ohlc.values()), pd.Timestamp(config["sample"]["end_date"]))
print("painel:", common_start.date(), "->", end.date())

painel: 2007-06-29 -> 2024-03-28


In [3]:
aligned_native = {ticker: align_and_fill_ohlc(df, common_start, end) for ticker, df in core_ohlc.items()}

# soja/café (USD) -> BRL-equivalente, usando o câmbio do próprio dia já alinhado/tratado
fx_close_brl_per_usd = aligned_native["BRL=X"]["Close"]
aligned_ohlc = dict(aligned_native)
for ticker in futures_tickers:
    converted = convert_ohlc_to_brl(aligned_native[ticker], fx_close_brl_per_usd)
    # A fonte só tem fechamento: as colunas ausentes entram vazias para manter o mesmo
    # formato dos demais ativos; as features que dependem delas saem logo abaixo.
    aligned_ohlc[ticker] = converted.reindex(columns=["Open", "High", "Low", "Close", "Volume"])

# Mesma regra dos preços: o VIX não é cotado em feriados americanos com a B3 aberta.
# O preenchimento causal de até 3 pregões preserva as janelas de lookback
# que seriam interrompidas por uma ausência isolada.
b3_days = get_b3_trading_days(common_start, end)
implied_vol_aligned = implied_vol.reindex(b3_days).ffill(limit=3)

## Cálculo das variáveis por ativo

In [4]:
fx_tickers = [a["ticker"] for a in config["assets"] if a["category"] == "fx"]
# Features que não existem para cada tipo de ativo (em vez de deixar colunas inteiras vazias):
# câmbio não tem volume centralizado; os futuros da base pública só têm fechamento.
NOT_APPLICABLE = {
    **{t: ["volume_normalized"] for t in fx_tickers},
    **{t: ["garman_klass_vol_21d", "garman_klass_vol_63d", "corwin_schultz_spread", "volume_normalized"] for t in futures_tickers},
}

features_by_asset = {}
for ticker, ohlc in aligned_ohlc.items():
    asset_features = build_asset_features(
        ohlc,
        lags=tuple(config["features"]["returns_lags"]),
        windows=tuple(config["features"]["rolling_windows"]),
    )
    asset_features = asset_features.drop(columns=NOT_APPLICABLE.get(ticker, []))
    features_by_asset[ticker] = pd.concat([asset_features, implied_vol_aligned], axis=1)

features_by_asset["SLCE3.SA"].tail()

,return_1d,return_5d,return_21d,realized_vol_21d,realized_vol_63d,garman_klass_vol_21d,garman_klass_vol_63d,garch_cond_vol,skew_21d,kurtosis_21d,skew_63d,kurtosis_63d,corwin_schultz_spread,volume_normalized,implied_vol_us,implied_vol_br
2024-03-22,0.004570,0.040315,0.007628,0.139491,0.237518,0.285253,0.272125,0.016247,0.065955,-0.814078,0.359436,1.072329,0.007642,0.910768,13.06,15.2659
2024-03-25,-0.009160,0.029048,0.009761,0.137425,0.234513,0.287070,0.273563,0.016025,0.091414,-0.777841,0.379719,1.202144,0.006494,0.803259,13.19,14.9130
2024-03-26,0.001533,0.020630,0.011294,0.137462,0.234193,0.286047,0.273242,0.015945,0.063504,-0.783976,0.398369,1.235282,0.003070,0.858345,13.24,14.4917
2024-03-27,0.004075,0.017436,0.000509,0.128058,0.232269,0.304905,0.280136,0.015717,-0.014707,-0.597231,0.430133,1.384759,0.000000,2.168101,12.78,14.0443
2024-03-28,0.005577,0.006594,0.003038,0.129106,0.232476,0.309066,0.283104,0.015516,-0.048714,-0.675474,0.414434,1.360031,0.018729,0.989843,13.01,13.9111


In [5]:
features_panel = pd.concat(features_by_asset, axis=1)
features_panel.to_parquet(PROCESSED_DIR / "features.parquet")
print("features.parquet salvo:", features_panel.shape)

features.parquet salvo: (4146, 103)


## Retorno agregado da carteira

Os log-retornos dos sete ativos, expressos em reais, são convertidos em retornos
simples antes da média com pesos iguais. O agregado volta à escala logarítmica
por `log1p`, com rebalanceamento diário e exigência dos sete componentes.
Sua volatilidade realizada é calculada sobre o retorno agregado, preservando
o efeito das covariâncias. Essas séries fornecem as observações do HMM.

In [6]:
from src.data.features import rolling_realized_volatility

asset_returns = pd.concat({t: df["return_1d"] for t, df in features_by_asset.items()}, axis=1)
portfolio_return_1d = equal_weighted_portfolio_return(asset_returns)
portfolio_vol = rolling_realized_volatility(portfolio_return_1d, windows=tuple(config["features"]["rolling_windows"]))
portfolio_series = pd.concat([portfolio_return_1d, portfolio_vol], axis=1)
portfolio_series.to_parquet(PROCESSED_DIR / "portfolio_return.parquet")
pd.DataFrame({"n_available": asset_returns.notna().sum(axis=1),
              "n_required": asset_returns.shape[1],
              "portfolio_valid": portfolio_return_1d.notna()}).to_csv(TABLES_DIR / "portfolio_coverage.csv")
features_panel.isna().mean().rename("missing_fraction").to_csv(TABLES_DIR / "feature_missingness.csv")
portfolio_series.tail()


,return_1d,realized_vol_21d,realized_vol_63d
2024-03-22,-0.013997,0.154832,0.138326
2024-03-25,0.002428,0.149277,0.137861
2024-03-26,-0.002924,0.142462,0.137405
2024-03-27,-0.002692,0.123120,0.137495
2024-03-28,0.021830,0.142522,0.144366


## Diagnósticos de estacionariedade, correlação e VIF

In [7]:
stationarity_by_asset = pd.concat(
    {ticker: stationarity_report(df) for ticker, df in features_by_asset.items()},
    names=["asset", "feature"],
)

# Correlação e VIF usam o painel empilhado, pois não dependem da ordem temporal.
# ADF e KPSS são aplicados separadamente a cada série.
pooled = pd.concat(features_by_asset.values(), axis=0)
pooled_complete = pooled.dropna()
print(
    "painel pooled (correlação/VIF, diagnóstico exploratório na amostra completa):",
    pooled_complete.shape, "de", pooled.shape[0], "linhas totais antes do dropna",
)

corr = correlation_matrix(pooled_complete)
kept_features, vif_report = vif_filter(pooled_complete, threshold=config["features"]["vif_threshold"])

stationarity_by_asset

painel pooled (correlação/VIF, diagnóstico exploratório na amostra completa): (8700, 16) de 29022 linhas totais antes do dropna


adf_stat    adf_pvalue  adf_rejects_unit_root  \
asset    feature                                                            
SLCE3.SA return_1d        -47.532105  0.000000e+00                   True   
         return_5d        -10.434549  1.578793e-18                   True   
         return_21d        -8.542768  9.682521e-14                   True   
         realized_vol_21d  -4.699537  8.432164e-05                   True   
         realized_vol_63d  -4.319215  4.113826e-04                   True   
...                              ...           ...                    ...   
KC=F     kurtosis_21d     -10.218618  5.398288e-18                   True   
         skew_63d          -6.044242  1.323529e-07                   True   
         kurtosis_63d      -7.974145  2.730876e-12                   True   
         implied_vol_us    -5.145132  1.136709e-05                   True   
         implied_vol_br    -5.233777  7.498292e-06                   True   

                           kpss_stat  kpss_pvalue  \
asset    feature                                    
SLCE3.SA return_1d          0.114644     0.100000   
         return_5d          0.129165     0.100000   
         return_21d         0.149020     0.100000   
         realized_vol_21d   1.234088     0.010000   
         realized_vol_63d   1.415902     0.010000   
...                              ...          ...   
KC=F     kurtosis_21d       0.511669     0.039039   
         skew_63d           1.087035     0.010000   
         kurtosis_63d       0.488456     0.044267   
         implied_vol_us     1.390230     0.010000   
         implied_vol_br     0.354408     0.096807   

                           kpss_fails_to_reject_stationary  \
asset    feature                                             
SLCE3.SA return_1d                                    True   
         return_5d                                    True   
         return_21d                                   True   
         realized_vol_21d                            False   
         realized_vol_63d                            False   
...                                                    ...   
KC=F     kurtosis_21d                                False   
         skew_63d                                    False   
         kurtosis_63d                                False   
         implied_vol_us                              False   
         implied_vol_br                               True   

                           stationary_both_tests  
asset    feature                                  
SLCE3.SA return_1d                          True  
         return_5d                          True  
         return_21d                         True  
         realized_vol_21d                  False  
         realized_vol_63d                  False  
...                                          ...  
KC=F     kurtosis_21d                      False  
         skew_63d                          False  
         kurtosis_63d                      False  
         implied_vol_us                    False  
         implied_vol_br                     True  

[103 rows x 7 columns]

In [8]:
vif_report

,status,vif
feature,,
realized_vol_63d,dropped,13.992931
realized_vol_21d,dropped,11.239535
return_1d,kept,1.262088
return_5d,kept,1.585522
return_21d,kept,1.439809
garman_klass_vol_21d,kept,5.149500
garman_klass_vol_63d,kept,2.844184
garch_cond_vol,kept,3.486646
skew_21d,kept,1.443934


In [9]:
stationarity_by_asset.to_csv(TABLES_DIR / "feature_stationarity.csv")
corr.to_csv(TABLES_DIR / "feature_correlation.csv")
vif_report.to_csv(TABLES_DIR / "feature_vif_selection.csv")

print(f"Features retidas após VIF (<= {config['features']['vif_threshold']}) — diagnóstico exploratório, não a seleção final:")
print(kept_features)
print()
print("Features descartadas por multicolinearidade:")
print(vif_report.index[vif_report["status"] == "dropped"].tolist())

Features retidas após VIF (<= 10) — diagnóstico exploratório, não a seleção final:
['return_1d', 'return_5d', 'return_21d', 'garman_klass_vol_21d', 'garman_klass_vol_63d', 'garch_cond_vol', 'skew_21d', 'kurtosis_21d', 'skew_63d', 'kurtosis_63d', 'corwin_schultz_spread', 'volume_normalized', 'implied_vol_us', 'implied_vol_br']

Features descartadas por multicolinearidade:
['realized_vol_63d', 'realized_vol_21d']
